# SynBioCrow_M2_4_8_RETROPATH_RR02_RESTORE

Restore the user-provided **older RR02 RetroRules** converter rather than TraceLD’s 17 example rules. Uses DOI 10.5281/zenodo.5828017 and a bounded 4,500-rule pilot. Targets: 3-hydroxypropionic acid and butanal. Native results are candidates only; raw rows are not assumed target-specific. This notebook requires network access and may consume significant memory/time.


In [ ]:
%pip -q install rdkit requests

In [ ]:
from pathlib import Path
ROOT=Path('/content/retropath2_standalone'); ROOT.mkdir(parents=True,exist_ok=True)
runner='import argparse,csv,json,os,pathlib,platform,subprocess,sys,time,urllib.request,zipfile,tarfile,shutil,hashlib\nROOT=pathlib.Path(\'/content/retropath2_standalone\'); ROOT.mkdir(parents=True,exist_ok=True)\nLOG=ROOT/\'logs\'; LOG.mkdir(exist_ok=True)\ndef fetch(url,dest):\n    req=urllib.request.Request(url,headers={\'User-Agent\':\'SynBioCrow-RetroPath2-Standalone\',\'Accept\':\'application/vnd.github+json\'})\n    with urllib.request.urlopen(req,timeout=90) as r: data=r.read()\n    pathlib.Path(dest).write_bytes(data); return data\ndef run(cmd,cwd=None,timeout=1200,log=\'command.log\'):\n    print(\'RUN\',cmd,flush=True)\n    with (LOG/log).open(\'w\') as f:\n        proc=subprocess.Popen(cmd,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)\n        try:\n            for line in proc.stdout: print(line,end=\'\',flush=True);f.write(line);f.flush()\n            code=proc.wait(timeout=timeout)\n        except Exception:\n            proc.kill();proc.wait();raise\n    if code:raise RuntimeError(\'Command failed (%d): %s\'%(code,cmd))\ndef install():\n    repo=ROOT/\'upstream\';\n    if not (repo/\'example_data\'/\'rules.csv\').exists():\n        archive=ROOT/\'upstream.zip\';fetch(\'https://github.com/TraceLD/retropath/archive/refs/heads/main.zip\',archive)\n        with zipfile.ZipFile(archive) as z:\n            z.extractall(ROOT/\'repo_unpack\')\n        dirs=list((ROOT/\'repo_unpack\').glob(\'retropath-*\'));assert dirs, \'Missing upstream directory\'\n        shutil.copytree(dirs[0],repo,dirs_exist_ok=True)\n    # Explicit upstream prerelease: GitHub /releases/latest returns 404 when a repo has only prereleases.\n    release=json.loads(fetch(\'https://api.github.com/repos/TraceLD/retropath/releases/tags/v0.0.1-alpha\',ROOT/\'release.json\'))\n    if release.get(\'tag_name\')!=\'v0.0.1-alpha\':\n        raise RuntimeError(\'Unexpected RetroPath release tag: \'+str(release.get(\'tag_name\')))\n    assets=release.get(\'assets\',[])\n    linux=[a for a in assets if (\'linux\' in a[\'name\'].lower() or \'ubuntu\' in a[\'name\'].lower()) and a[\'name\'].lower().endswith((\'.zip\',\'.tar.gz\',\'.tgz\')) and (\'arm\' not in a[\'name\'].lower())]\n    if not linux: raise RuntimeError(\'No Linux x64 archive in latest GitHub release. Available: \'+repr([a[\'name\'] for a in assets]))\n    chosen=linux[0]; name=chosen[\'name\']; arc=ROOT/name\n    if not (arc.exists() and zipfile.is_zipfile(arc)):\n        fetch(chosen[\'browser_download_url\'],arc)\n    if not zipfile.is_zipfile(arc):\n        raise RuntimeError(\'Release download was not a valid ZIP: \'+str(arc))\n    dest=ROOT/\'distribution\';dest.mkdir(exist_ok=True)\n    if name.endswith(\'.zip\'):\n        with zipfile.ZipFile(arc) as z:z.extractall(dest)\n    else:\n        with tarfile.open(arc) as z:z.extractall(dest,filter=\'data\')\n    candidates=list(dest.rglob(\'RetroPath.Cli\'))\n    if not candidates: raise RuntimeError(\'Release has no RetroPath.Cli executable\')\n    cli=candidates[0];cli.chmod(cli.stat().st_mode|0o111)\n    (ROOT/\'install.json\').write_text(json.dumps({\'release\':release.get(\'tag_name\'),\'prerelease\':release.get(\'prerelease\'),\'release_url\':release.get(\'html_url\'),\'asset\':name,\'sha256\':hashlib.sha256(arc.read_bytes()).hexdigest(),\'cli\':str(cli)},indent=2))\n    print(\'INSTALLED\',cli)\ndef target_excluded_sink(original, destination, target_inchi):\n    """Remove source/sink InChI collisions while preserving all other native sink rows."""\n    with original.open(newline=\'\', encoding=\'utf-8-sig\') as fh:\n        reader=csv.DictReader(fh)\n        fields=reader.fieldnames\n        if not fields or not any(\'inchi\' in col.lower() for col in fields):\n            raise ValueError(\'Sink CSV lacks an InChI column: \'+str(fields))\n        col=next(col for col in fields if col.lower().strip()==\'inchi\')\n        records=list(reader)\n    # Native parser normalizes protonation layers; /p-1 of 3-HP collides\n    # with the neutral target even when raw InChI strings differ.\n    def base_identity(inchi):\n        return inchi.strip().split(\'/p\', 1)[0]\n    target_base=base_identity(target_inchi)\n    kept=[r for r in records if base_identity(r[col])!=target_base]\n    removed=len(records)-len(kept)\n    removed_entries=[{\'name\':r.get(\'Name\',\'\'),\'inchi\':r[col]} for r in records if base_identity(r[col])==target_base]\n\n    with destination.open(\'w\', newline=\'\', encoding=\'utf-8\') as fh:\n        writer=csv.DictWriter(fh,fieldnames=fields)\n        writer.writeheader();writer.writerows(kept)\n    if not kept: raise ValueError(\'Target-specific sink filtering left no sink metabolites\')\n    return {\'original_sink_rows\':len(records),\'filtered_sink_rows\':len(kept),\'target_overlaps_removed\':removed,\'removed_entries\':removed_entries,\'comparison\':\'InChI excluding protonation /p layer\'}\n\n\n# Official pre-parsed RetroRules rr02 RetroPath2-compatible explicit-H dataset.\nRR02_RECORD=\'5828017\'\nRR02_FILE=\'retrorules_rr02_rp2_hs.tar.gz\'\nREQUIRED_RULE_COLUMNS=[\'Rule ID\',\'Rule\',\'EC number\',\'Reaction order\',\'Diameter\',\'Score\']\ndef download_rr02():\n    import urllib.error\n    import requests\n    meta=requests.get(\'https://zenodo.org/api/records/\'+RR02_RECORD,timeout=90)\n    meta.raise_for_status();record=meta.json()\n    files=record.get(\'files\',[])\n    match=next((f for f in files if f.get(\'key\')==RR02_FILE),None)\n    if not match:raise RuntimeError(\'Official Zenodo rr02 archive not found: \'+str([f.get(\'key\') for f in files]))\n    url=match[\'links\'][\'self\'];size=int(match.get(\'size\',0))\n    path=ROOT/RR02_FILE\n    if not path.exists() or (size and path.stat().st_size!=size):\n        temp=path.with_suffix(\'.partial\')\n        with requests.get(url,stream=True,timeout=(45,180)) as response:\n            response.raise_for_status()\n            with temp.open(\'wb\') as f:\n                done=0\n                for chunk in response.iter_content(chunk_size=2**20):\n                    if chunk:\n                        f.write(chunk);done+=len(chunk)\n                        if done//(10*2**20)!=(done-len(chunk))//(10*2**20):print(\'RR02 download\',round(done/2**20,1),\'MiB\',flush=True)\n        temp.replace(path)\n    if size and path.stat().st_size!=size:raise RuntimeError(\'Zenodo download size mismatch\')\n    digest=hashlib.md5(path.read_bytes()).hexdigest()\n    # MD5 checksum published with the original rr02 dataset.\n    expected=(match.get(\'checksum\') or \'\').split(\':\')[-1]\n    if expected and len(expected)==32 and digest!=expected:raise RuntimeError(\'Zenodo checksum mismatch\')\n    (ROOT/\'rr02_provenance.json\').write_text(json.dumps({\'zenodo_record\':RR02_RECORD,\'doi\':\'10.5281/zenodo.5828017\',\'filename\':RR02_FILE,\'download_url\':url,\'size\':path.stat().st_size,\'md5\':digest,\'record_checksum\':match.get(\'checksum\'),\'dataset\':\'rr02 RetroPath2 pre-parsed explicit-H\'},indent=2))\n    return path\n\ndef build_rules(diameters=(2,4),limit=4500):\n    import io,gzip,collections\n    archive=download_rr02()\n    dest=ROOT/\'rr02_rules\';dest.mkdir(exist_ok=True)\n    output=dest/(\'rules_d\'+\'_\'.join(map(str,diameters))+\'_max\'+str(limit)+\'.csv\')\n    manifest=dest/\'rule_manifest.json\'\n    accepted=0;seen=set();counts=collections.Counter();sources=[];invalid=collections.Counter()\n    # The archive may contain multiple diameter-specific CSV files. Process each\n    # member as a stream without unpacking the entire archive into Colab storage.\n    with output.open(\'w\',newline=\'\',encoding=\'utf-8\') as of:\n        writer=csv.DictWriter(of,fieldnames=REQUIRED_RULE_COLUMNS);writer.writeheader()\n        with tarfile.open(archive,\'r:gz\') as tar:\n            members=[m for m in tar if m.isfile() and (m.name.lower().endswith((\'.csv\',\'.tsv\',\'.csv.gz\',\'.tsv.gz\')))]\n            if not members:raise RuntimeError(\'RR02 archive contains no CSV/TSV members; inspect archive before proceeding\')\n            for member in members:\n                if accepted>=limit:break\n                print(\'Inspecting rule member:\',member.name,flush=True)\n                raw=tar.extractfile(member)\n                if raw is None:continue\n                if member.name.lower().endswith(\'.gz\'):raw=gzip.GzipFile(fileobj=raw)\n                stream=io.TextIOWrapper(raw,encoding=\'utf-8-sig\',errors=\'replace\',newline=\'\')\n                delimiter=\'\\t\' if \'.tsv\' in member.name.lower() else \',\'\n                reader=csv.DictReader(stream,delimiter=delimiter)\n                headers=reader.fieldnames or []\n                canonical={h.strip().lower().replace(\'_\',\' \'):h for h in headers}\n                aliases={\'Rule ID\':[\'rule id\',\'rule_id\',\'id\'],\'Rule\':[\'rule\',\'rule smarts\',\'smarts\',\'smirks\'],\'EC number\':[\'ec number\',\'ec\',\'ec number(s)\'],\'Reaction order\':[\'reaction order\',\'order\'],\'Diameter\':[\'diameter\'],\'Score\':[\'score\',\'rule score\']}\n                mapping={field:next((canonical[a] for a in names if a in canonical),None) for field,names in aliases.items()}\n                if not mapping[\'Rule\'] or not mapping[\'Rule ID\'] or not mapping[\'Diameter\']:\n                    invalid[\'unsupported_member_schema\']+=1\n                    print(\'SKIP incompatible member headers\',headers[:20],flush=True);continue\n                count_member=0\n                for row in reader:\n                    try:diam=int(float(row[mapping[\'Diameter\']]))\n                    except (ValueError,TypeError):invalid[\'invalid_diameter\']+=1;continue\n                    if diam not in diameters:continue\n                    rule=(row.get(mapping[\'Rule\']) or \'\').strip()\n                    if \'>>\' not in rule:invalid[\'invalid_smarts\']+=1;continue\n                    rule_id=(row.get(mapping[\'Rule ID\']) or \'\').strip()\n                    if not rule_id:invalid[\'missing_id\']+=1;continue\n                    key=(rule_id,rule,diam)\n                    if key in seen:continue\n                    seen.add(key)\n                    record={\'Rule ID\':rule_id,\'Rule\':rule,\'EC number\':row.get(mapping[\'EC number\'],\'\') if mapping[\'EC number\'] else \'\', \'Reaction order\':row.get(mapping[\'Reaction order\'],\'1\') if mapping[\'Reaction order\'] else \'1\',\'Diameter\':diam,\'Score\':row.get(mapping[\'Score\'],\'1000\') if mapping[\'Score\'] else \'1000\'}\n                    writer.writerow(record);accepted+=1;count_member+=1;counts[diam]+=1\n                    if accepted%5000==0:print(\'Accepted\',accepted,\'native-format rules\',flush=True)\n                    if accepted>=limit:break\n                sources.append({\'member\':member.name,\'accepted\':count_member,\'headers\':headers})\n    if accepted<100:raise RuntimeError(\'Too few compatible rules: \'+str(accepted)+\'; inspect rr02 archive and manifest\')\n    audit={\'rules\':accepted,\'diameters\':list(diameters),\'limit\':limit,\'counts_by_diameter\':dict(counts),\'members\':sources,\'skips\':dict(invalid),\'rule_file\':str(output),\'sha256\':hashlib.sha256(output.read_bytes()).hexdigest(),\'source_doi\':\'10.5281/zenodo.5828017\',\'explicit_hydrogens\':True,\'schema\':REQUIRED_RULE_COLUMNS,\'truncated\':accepted>=limit}\n    manifest.write_text(json.dumps(audit,indent=2));print(\'RR02 RULE AUDIT\',json.dumps(audit,indent=2),flush=True)\n    return output\n\ndef execute(targets,steps=4,timeout=1200):\n    from rdkit import Chem\n    inst=json.loads((ROOT/\'install.json\').read_text());cli=pathlib.Path(inst[\'cli\'])\n    example=ROOT/\'upstream\'/\'example_data\';rules=build_rules(diameters=(2,4),limit=4500);sink=example/\'sink.csv\'\n    # The upstream small rule collection is genuine, but has narrow coverage.\n    # Never use upstream source.csv (pinocembrin) as a replacement for target inputs.\n    assert rules.is_file() and sink.is_file()\n    summary=[]\n    for name,smiles in targets:\n        work=ROOT/\'runs\'/name;work.mkdir(parents=True,exist_ok=True)\n        mol=Chem.MolFromSmiles(smiles)\n        if mol is None:raise ValueError(\'Invalid target SMILES \'+smiles)\n        inchi=Chem.MolToInchi(mol)\n        source=work/\'source.csv\'\n        with source.open(\'w\',newline=\'\') as f:\n            w=csv.writer(f);w.writerow([\'Name\',\'InChI\']);w.writerow([name,inchi])\n        assert \'pinocembrin\' not in source.read_text().lower()\n        target_sink=work/\'sink_target_filtered.csv\'\n        sink_audit=target_excluded_sink(sink,target_sink,inchi)\n        (work/\'sink_filter_audit.json\').write_text(json.dumps(sink_audit,indent=2))\n        if name==\'3_hydroxypropionic_acid\' and sink_audit[\'target_overlaps_removed\']==0:\n            raise RuntimeError(\'3-HP target protonation collision was not removed; stop before native execution\')\n        start=time.time(); status=\'NATIVE_FAILED\';error=None\n        try:\n            # Execute from per-target working directory to isolate native results folder.\n            run([str(cli),str(rules),str(source),str(target_sink),str(steps)],cwd=work,timeout=timeout,log=name+\'.log\')\n            status=\'NATIVE_COMPLETED\'\n        except Exception as exc:error=str(exc)\n        resultfiles=list(work.rglob(\'results.csv\'))\n        rows=[]\n        for path in resultfiles:\n            with path.open(newline=\'\') as f:rows.extend(list(csv.DictReader(f)))\n        # A completed search with zero rows is not a discovered pathway.\n        actual=[r for r in rows if name.lower() in \' \'.join(str(v) for v in r.values()).lower()]\n        if rows and not actual:\n            status=\'OUTPUT_TARGET_IDENTITY_UNVERIFIED\'\n        report={\'engine\':\'TraceLD/retropath\',\'target\':name,\'target_smiles\':smiles,\'target_inchi\':inchi,\'status\':status,\'native_results\':len(rows),\'target_named_results\':len(actual),\'target_specific_reactions_verified\':False,\'source_file\':str(source),\'rule_file\':str(rules),\'sink_file\':str(target_sink),\'original_sink_file\':str(sink),\'sink_filter_audit\':sink_audit,\'rules_provenance\':\'Zenodo rr02-rp2-hs RetroPath2-ready rules; DOI 10.5281/zenodo.5828017; explicit H; diameter 2,4; capped at 4500; see rule_manifest.json\',\'elapsed_s\':round(time.time()-start,2),\'error\':error,\'result_files\':[str(x) for x in resultfiles]}\n        (work/\'run_status.json\').write_text(json.dumps(report,indent=2));summary.append(report);print(json.dumps(report,indent=2),flush=True)\n    (ROOT/\'summary.json\').write_text(json.dumps(summary,indent=2))\n    with zipfile.ZipFile(ROOT/\'SynBioCrow_M2_4_8_RETROPATH_RR02_RESTORE_ALL_OUTPUTS.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\n        for f in ROOT.rglob(\'*\'):\n            if f.is_file() and f.name!=\'SynBioCrow_M2_4_8_RETROPATH_RR02_RESTORE_ALL_OUTPUTS.zip\' and \'repo_unpack\' not in f.parts and \'distribution\' not in f.parts and \'upstream\' not in f.parts and f.name!=RR02_FILE and not (f.suffix==\'.csv\' and \'rr02_rules\' in f.parts):\n                z.write(f,f.relative_to(ROOT))\n    print(\'RESULT_ARCHIVE\',ROOT/\'SynBioCrow_M2_4_8_RETROPATH_RR02_RESTORE_ALL_OUTPUTS.zip\')\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'command\',choices=[\'install\',\'run\',\'rules\']);parser.add_argument(\'--steps\',type=int,default=4);parser.add_argument(\'--timeout\',type=int,default=1200);a=parser.parse_args()\n    if a.command==\'install\':install()\n    elif a.command==\'rules\':build_rules(diameters=(2,4),limit=4500)\n    else:execute([(\'3_hydroxypropionic_acid\',\'O=C(O)CCO\'),(\'butanal\',\'CCCC=O\')],a.steps,a.timeout)\n'
(ROOT/'retropath2_runner.py').write_text(runner)
print('Recovered rr02 runner',len(runner),'characters')

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,str(ROOT/'retropath2_runner.py'),'install'],check=True)

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,str(ROOT/'retropath2_runner.py'),'rules'],check=True)

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,str(ROOT/'retropath2_runner.py'),'run','--steps','1','--timeout','1200'],check=True)

In [ ]:
from pathlib import Path
import json
print(json.dumps(json.loads((ROOT/'summary.json').read_text()),indent=2)[:12000])
print('OUTPUT',ROOT/'SynBioCrow_M2_4_8_RETROPATH_RR02_RESTORE_ALL_OUTPUTS.zip')

In [ ]:
# Rescue download (safe to rerun after execution)
from google.colab import files
p=ROOT/'SynBioCrow_M2_4_8_RETROPATH_RR02_RESTORE_ALL_OUTPUTS.zip'
if p.exists(): files.download(str(p))
else: print('No result archive yet; inspect earlier cell errors')